# 3.3 极化变量的统计定义

**学习目标**
- 理解极化变量的集合平均定义
- 掌握功率和相关系数的统计计算方法
- 理解各极化变量之间的关系
- 观察粒子谱和取向分布对极化变量的综合影响

**参考文献**：Ryzhkov & Zrnic, Chapter 3, pp. 311-350

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets
plt.rcParams['font.family'] = ['DejaVu Sans', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

## 1. 理论背景

### 协方差矩阵

对于双线偏振雷达，后向散射的统计特性由协方差矩阵描述：

$$\mathbf{C} = \begin{pmatrix} \langle S_{hh}S_{hh}^* \rangle & \langle S_{hh}S_{vv}^* \rangle \\ \langle S_{vv}S_{hh}^* \rangle & \langle S_{vv}S_{vv}^* \rangle \end{pmatrix}$$

### 极化变量的定义

- **反射率**: $Z = \frac{4\lambda^4}{\pi^4 |K|^2} \langle S_{hh}S_{hh}^* \rangle$
- **差分反射率**: $Z_{DR} = 10\log_{10}\frac{\langle S_{hh}S_{hh}^* \rangle}{\langle S_{vv}S_{vv}^* \rangle}$
- **相关系数**: $\rho_{hv} = \frac{\langle S_{hh}S_{vv}^* \rangle}{\sqrt{\langle S_{hh}S_{hh}^* \rangle \langle S_{vv}S_{vv}^* \rangle}}$

In [ ]:
def marshall_palmer_dsd(D, N0=8e6, Lambda=4.1):
    """
    Marshall-Palmer 滴谱分布
    D: 直径 (m)
    N0: 截距参数 (mm^-1 m^-3)
    Lambda: 斜率参数 (mm^-1)
    """
    D_mm = D * 1000  # convert to mm
    return N0 * np.exp(-Lambda * D_mm)

def gamma_dsd(D, N0=5e6, Lambda=3.0, mu=2):
    """
    Gamma 滴谱分布
    mu: 形状参数
    """
    D_mm = D * 1000
    return N0 * D_mm**mu * np.exp(-Lambda * D_mm)

def calculate_reflectivity(D, N_D, wavelength, K_sq=0.93):
    """
    计算反射率因子 Z
    D: 直径数组
    N_D: 滴谱分布
    """
    D_mm = D * 1000
    Z = np.sum(N_D * D_mm**6) * 1e-3  # mm^6/m^3 -> mm^6/m^3
    return Z

def calculate_polarimetric_variables(D, N_D, wavelength, sigma_theta_deg=15, 
                                    dD=0.0001):
    """
    计算极化变量
    简化模型：假设雨滴为椭球，倾角为高斯分布
    """
    m_water = 1.33 + 0.001j
    K_sq = np.abs((m_water**2 - 1) / (m_water**2 + 2))**2
    sigma_theta = np.radians(sigma_theta_deg)
    
    # 倾角分布积分
    theta_range = np.linspace(-np.pi/2, np.pi/2, 500)
    p_theta = np.exp(-theta_range**2 / (2*sigma_theta**2)) / (sigma_theta * np.sqrt(2*np.pi))
    
    # 计算协方差元素
    lambda_m = wavelength * 1000  # mm
    
    # 瑞利散射截面
    def sigma_rayleigh(D_mm):
        return (np.pi**5 / lambda_m**4) * K_sq * D_mm**6
    
    # 形状因子（椭球近似）
    axis_ratio = 0.6  # 雨滴典型轴比
    f_hh = np.cos(theta_range)**2 * (1 + axis_ratio**2) / 2 + np.sin(theta_range)**2 * (1 - axis_ratio**2)
    f_vv = np.sin(theta_range)**2 * (1 + axis_ratio**2) / 2 + np.cos(theta_range)**2 * (1 - axis_ratio**2)
    
    # 积分计算协方差
    integrand_hh = np.zeros_like(theta_range)
    integrand_vv = np.zeros_like(theta_range)
    
    for i, theta in enumerate(theta_range):
        for j, D_j in enumerate(D):
            D_mm = D_j * 1000
            weight = N_D[j] * dD
            sigma = sigma_rayleigh(D_mm)
            integrand_hh[i] += sigma * f_hh[i] * weight
            integrand_vv[i] += sigma * f_vv[i] * weight
    
    cov_hh = np.trapz(integrand_hh * p_theta, theta_range)
    cov_vv = np.trapz(integrand_vv * p_theta, theta_range)
    cov_hh_vv = np.sqrt(cov_hh * cov_vv) * 0.95  # 简化假设
    
    # 计算极化变量
    Z_hh = 4.343 * np.log10(cov_hh + 1e-10) + 30  # dBZ
    Z_vv = 4.343 * np.log10(cov_vv + 1e-10) + 30
    ZDR = Z_hh - Z_vv
    rho_hv = np.abs(cov_hh_vv) / np.sqrt(cov_hh * cov_vv + 1e-10)
    
    return {'Z': Z_hh, 'ZDR': ZDR, 'rho_hv': rho_hv, 'cov_hh': cov_hh, 'cov_vv': cov_vv}

## 2. 滴谱分布与极化变量

In [ ]:
def plot_dsd_polarimetric(R=10.0, dsd_type='exponential', wavelength=0.107):
    """可视化滴谱分布和极化变量的关系"""
    
    # 直径范围
    D = np.linspace(0.1, 6, 200) / 1000  # m
    dD = D[1] - D[0]
    
    # 根据降雨率计算Lambda
    if dsd_type == 'exponential':
        Lambda = 4.1 * R**(-0.21)  # mm^-1
        N_D = marshall_palmer_dsd(D, Lambda=Lambda)
        dsd_label = 'Marshall-Palmer'
    else:  # gamma
        Lambda = 3.0 * R**(-0.18)
        N_D = gamma_dsd(D, Lambda=Lambda, mu=2)
        dsd_label = 'Gamma (μ=2)'
    
    # 计算极化变量
    pol_vars = calculate_polarimetric_variables(D, N_D, wavelength, sigma_theta_deg=15, dD=dD)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 子图1: 滴谱分布
    ax1 = axes[0, 0]
    D_mm = D * 1000
    ax1.semilogy(D_mm, N_D, 'b-', linewidth=2)
    ax1.set_xlabel('直径 D (mm)', fontsize=11)
    ax1.set_ylabel('N(D) (mm⁻¹ m⁻³)', fontsize=11)
    ax1.set_title(f'{dsd_label} 滴谱分布 (R={R} mm/h)', fontsize=12)
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0, 6)
    ax1.set_ylim(1, 1e6)
    
    # 子图2: D^6权重（反映对Z的贡献）
    ax2 = axes[0, 1]
    contribution = N_D * D_mm**6
    ax2.plot(D_mm, contribution / contribution.max(), 'r-', linewidth=2)
    ax2.axvline(x=1.5, color='green', linestyle='--', label='峰值位置')
    ax2.set_xlabel('直径 D (mm)', fontsize=11)
    ax2.set_ylabel('归一化贡献', fontsize=11)
    ax2.set_title('D⁶ 加权滴谱（反射率贡献）', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 子图3: 反射率 vs 降雨率
    ax3 = axes[1, 0]
    R_range = np.linspace(0.5, 100, 100)
    Z_range = []
    for R_val in R_range:
        if dsd_type == 'exponential':
            Lambda = 4.1 * R_val**(-0.21)
            N_D_val = marshall_palmer_dsd(D, Lambda=Lambda)
        else:
            Lambda = 3.0 * R_val**(-0.18)
            N_D_val = gamma_dsd(D, Lambda=Lambda, mu=2)
        pol = calculate_polarimetric_variables(D, N_D_val, wavelength, dD=dD)
        Z_range.append(pol['Z'])
    ax3.semilogx(R_range, Z_range, 'b-', linewidth=2)
    ax3.scatter([R], [pol_vars['Z']], color='red', s=100, zorder=5, label=f'当前: R={R}mm/h')
    ax3.set_xlabel('降雨率 R (mm/h)', fontsize=11)
    ax3.set_ylabel('反射率 Z (dBZ)', fontsize=11)
    ax3.set_title(f'R-Z 关系 ({dsd_label})', fontsize=12)
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 子图4: ZDR vs 降雨率
    ax4 = axes[1, 1]
    ZDR_range = []
    for R_val in R_range:
        if dsd_type == 'exponential':
            Lambda = 4.1 * R_val**(-0.21)
            N_D_val = marshall_palmer_dsd(D, Lambda=Lambda)
        else:
            Lambda = 3.0 * R_val**(-0.18)
            N_D_val = gamma_dsd(D, Lambda=Lambda, mu=2)
        pol = calculate_polarimetric_variables(D, N_D_val, wavelength, dD=dD)
        ZDR_range.append(pol['ZDR'])
    ax4.plot(R_range, ZDR_range, 'g-', linewidth=2)
    ax4.scatter([R], [pol_vars['ZDR']], color='red', s=100, zorder=5, label=f'当前: ZDR={pol_vars["ZDR"]:.1f}dB')
    ax4.set_xlabel('降雨率 R (mm/h)', fontsize=11)
    ax4.set_ylabel('差分反射率 ZDR (dB)', fontsize=11)
    ax4.set_title(f'降雨率-ZDR 关系', fontsize=12)
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n=== 当前参数计算结果 ===")
    print(f"降雨率: {R} mm/h")
    print(f"滴谱类型: {dsd_label}")
    print(f"波长: {wavelength*100:.0f} cm (S-band)")
    print(f"反射率 Z: {pol_vars['Z']:.1f} dBZ")
    print(f"差分反射率 ZDR: {pol_vars['ZDR']:.2f} dB")
    print(f"相关系数 ρhv: {pol_vars['rho_hv']:.4f}")

In [ ]:
interact_dsd = interact(plot_dsd_polarimetric,
    R=widgets.FloatSlider(min=0.5, max=100, step=0.5, value=10.0, 
                         description='降雨率 (mm/h)'),
    dsd_type=widgets.Dropdown(options=[('指数分布', 'exponential'), ('Gamma分布', 'gamma')],
                            value='exponential', description='滴谱类型'),
    wavelength=widgets.Dropdown(options=[('S-band 10.7cm', 0.107), ('C-band 5.4cm', 0.054), ('X-band 3.2cm', 0.032)],
                             value=0.107, description='雷达波长')
)

## 练习题

1. **滴谱影响**：Marshall-Palmer分布和Gamma分布的ZDR计算结果有何差异？

2. **波段效应**：不同波段雷达测得的ZDR是否相同？为什么？

3. **R-Z关系**：为什么R-Z关系存在较大离散度？这对降雨估计有什么影响？

4. **相关系数解释**：ρhv接近1表示什么物理含义？

## 参考文献

- Ryzhkov, A.V., and D.S. Zrnic, 2019: *Radar Polarimetry for Weather Observations*, Chapter 3, Springer.